<a href="https://colab.research.google.com/github/Krishnapabbu17/Wharton-DS-Project/blob/main/notebooks/01_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/Krishnapabbu17/Wharton-DS-Project.git

Cloning into 'Wharton-DS-Project'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 28 (delta 10), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 16.97 KiB | 2.12 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [ ]:
import sys
sys.path.append('/content/Wharton-DS-Project')

In [ ]:
!pip install pandas openpyxl

In [ ]:
!ls -lh data

ls: cannot access 'data': No such file or directory


In [ ]:
!pwd

/content


In [ ]:
%cd /content/Wharton-DS-Project

/content/Wharton-DS-Project


In [ ]:
!ls
!ls data

data  notebooks  README.md  src
'whl_2025 (1).xlsx'		   WHSDSC_Rnd1_matchups.xlsx
 WHSDSC_2026_DataDictionary.xlsx


In [ ]:
import pandas as pd

whl_path = "data/whl_2025 (1).xlsx"
matchups_path = "data/WHSDSC_Rnd1_matchups.xlsx"
dict_path = "data/WHSDSC_2026_DataDictionary.xlsx"

# See tabs
print("WHL tabs:", pd.ExcelFile(whl_path).sheet_names)
print("Matchups tabs:", pd.ExcelFile(matchups_path).sheet_names)
print("Dictionary tabs:", pd.ExcelFile(dict_path).sheet_names)

# Load first tab of each
whl = pd.read_excel(whl_path, sheet_name=0)
matchups = pd.read_excel(matchups_path, sheet_name=0)
dictionary = pd.read_excel(dict_path, sheet_name=0)

print("WHL shape:", whl.shape)
print("Matchups shape:", matchups.shape)
print("Dictionary shape:", dictionary.shape)

print("\nWHL columns:\n", whl.columns.tolist())
print("\nMatchups columns:\n", matchups.columns.tolist())

WHL tabs: ['whl_2025']
Matchups tabs: ['matchups']
Dictionary tabs: ['dictionary', 'Ex_game1']
WHL shape: (25827, 26)
Matchups shape: (16, 4)
Dictionary shape: (26, 6)

WHL columns:
 ['game_id', 'record_id', 'home_team', 'away_team', 'went_ot', 'home_off_line', 'home_def_pairing', 'away_off_line', 'away_def_pairing', 'home_goalie', 'away_goalie', 'toi', 'home_assists', 'home_shots', 'home_xg', 'home_max_xg', 'home_goals', 'away_assists', 'away_shots', 'away_xg', 'away_max_xg', 'away_goals', 'home_penalties_committed', 'home_penalty_minutes', 'away_penalties_committed', 'away_penalty_minutes']

Matchups columns:
 ['game', 'game_id', 'home_team', 'away_team']


In [ ]:
whl.head(3)

,game_id,record_id,home_team,away_team,went_ot,home_off_line,home_def_pairing,away_off_line,away_def_pairing,home_goalie,...,home_goals,away_assists,away_shots,away_xg,away_max_xg,away_goals,home_penalties_committed,home_penalty_minutes,away_penalties_committed,away_penalty_minutes
0,game_1,record_1,thailand,pakistan,0,PP_kill_dwn,PP_kill_dwn,PP_up,PP_up,player_id_142,...,0,2,9,1.4645,0.2166,1,7,14,1,2
1,game_1,record_2,thailand,pakistan,0,second_off,second_def,second_off,second_def,player_id_142,...,0,2,1,0.0928,0.0928,1,0,0,0,0
2,game_1,record_3,thailand,pakistan,0,first_off,second_def,second_off,second_def,player_id_142,...,0,0,2,0.1880,0.0940,0,0,0,0,0


In [ ]:
matchups

,game,game_id,home_team,away_team
0,1,game_1,brazil,kazakhstan
1,2,game_2,netherlands,mongolia
2,3,game_3,peru,rwanda
3,4,game_4,thailand,oman
4,5,game_5,pakistan,germany
5,6,game_6,india,usa
6,7,game_7,panama,switzerland
7,8,game_8,iceland,canada
8,9,game_9,china,france
9,10,game_10,philippines,morocco


In [ ]:
matchups.head(20)

,game,game_id,home_team,away_team
0,1,game_1,brazil,kazakhstan
1,2,game_2,netherlands,mongolia
2,3,game_3,peru,rwanda
3,4,game_4,thailand,oman
4,5,game_5,pakistan,germany
5,6,game_6,india,usa
6,7,game_7,panama,switzerland
7,8,game_8,iceland,canada
8,9,game_9,china,france
9,10,game_10,philippines,morocco


In [ ]:
matchups.columns.tolist()

['game', 'game_id', 'home_team', 'away_team']

In [2]:
# =========================
# WHSDC 2026 — Part 1A (Stable Strategy)
# Power rankings + matchup probabilities using z-scored team strength
# =========================

!pip -q install pandas openpyxl numpy

import pandas as pd
import numpy as np

# ---------- 1) Load data ----------
whl_path = "data/whl_2025 (1).xlsx"
matchups_path = "data/WHSDSC_Rnd1_matchups.xlsx"

whl = pd.read_excel(whl_path, sheet_name=0)
matchups = pd.read_excel(matchups_path, sheet_name=0)

# normalize columns
whl.columns = [str(c).strip().lower().replace(" ", "_") for c in whl.columns]
matchups.columns = [str(c).strip().lower().replace(" ", "_") for c in matchups.columns]

# ---------- 2) Identify key columns ----------
def pick_first(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

gid = pick_first(whl, ["game_id", "gameid", "id", "match_id", "matchid"])
date_col = pick_first(whl, ["date", "game_date", "gamedate"])

home_team = pick_first(whl, ["home_team", "hometeam", "home"])
away_team = pick_first(whl, ["away_team", "awayteam", "away"])
home_goals = pick_first(whl, ["home_goals", "goals_home", "homegoal"])
away_goals = pick_first(whl, ["away_goals", "goals_away", "awaygoal"])
home_xg = pick_first(whl, ["home_xg", "xg_home", "homexg"])
away_xg = pick_first(whl, ["away_xg", "xg_away", "awayxg"])

# If no game_id column, build one from date + teams (works if those exist)
if gid is None:
    if date_col is None or home_team is None or away_team is None:
        raise ValueError("No game_id found and can't build one (need date + home_team + away_team).")
    whl["_game_id"] = (
        whl[date_col].astype(str) + "__" +
        whl[home_team].astype(str) + "__" +
        whl[away_team].astype(str)
    )
    gid = "_game_id"

needed = [gid, home_team, away_team, home_goals, away_goals, home_xg, away_xg]
if any(c is None for c in needed):
    raise ValueError(f"Missing required columns. Found: {needed}")

# ---------- 3) Make game-level table (1 row per game) ----------
games = (
    whl
    .drop_duplicates(subset=[gid])
    [[gid, home_team, away_team, home_goals, away_goals, home_xg, away_xg]]
    .copy()
)

games.columns = ["game_id", "home_team", "away_team", "home_goals", "away_goals", "home_xg", "away_xg"]

print("Games:", games.shape, "| Teams in games:", pd.unique(pd.concat([games["home_team"], games["away_team"]])).shape[0])

# ---------- 4) Build team-game rows (2 rows per game) ----------
rows = []
for _, r in games.iterrows():
    rows.append({
        "team": r["home_team"],
        "goals_for": r["home_goals"],
        "goals_against": r["away_goals"],
        "xg_for": r["home_xg"],
        "xg_against": r["away_xg"],
        "win": 1 if r["home_goals"] > r["away_goals"] else 0
    })
    rows.append({
        "team": r["away_team"],
        "goals_for": r["away_goals"],
        "goals_against": r["home_goals"],
        "xg_for": r["away_xg"],
        "xg_against": r["home_xg"],
        "win": 1 if r["away_goals"] > r["home_goals"] else 0
    })

team_games = pd.DataFrame(rows)

# ---------- 5) Team season summary ----------
team_table = (
    team_games
    .groupby("team", as_index=False)
    .agg(
        games=("win", "count"),
        wins=("win", "sum"),
        avg_goals_for=("goals_for", "mean"),
        avg_goals_against=("goals_against", "mean"),
        avg_xg_for=("xg_for", "mean"),
        avg_xg_against=("xg_against", "mean"),
    )
)

team_table["goal_diff"] = team_table["avg_goals_for"] - team_table["avg_goals_against"]
team_table["xg_diff"] = team_table["avg_xg_for"] - team_table["avg_xg_against"]
team_table["win_pct"] = team_table["wins"] / team_table["games"]

print("Teams:", len(team_table))
display(team_table.head())

# ---------- 6) Build a FAIR strength score using z-scores ----------
# z = (value - mean) / std
def zscore(s):
    s = s.astype(float)
    return (s - s.mean()) / (s.std(ddof=0) + 1e-9)

team_table["z_xg"] = zscore(team_table["xg_diff"])
team_table["z_goal"] = zscore(team_table["goal_diff"])
team_table["z_win"] = zscore(team_table["win_pct"])

# You can keep equal weights (simple + defensible)
team_table["strength_score"] = (team_table["z_xg"] + team_table["z_goal"] + team_table["z_win"]) / 3.0

# Power rankings (32 teams)
power_rankings = team_table.sort_values("strength_score", ascending=False).reset_index(drop=True)
power_rankings["rank"] = power_rankings.index + 1

print("\nPower Rankings (Top 10):")
display(power_rankings[["rank", "team", "strength_score", "xg_diff", "goal_diff", "win_pct"]].head(10))

# ---------- 7) Matchup probabilities (16 matchups) ----------
# Find matchup home/away columns
m_home = pick_first(matchups, ["home_team", "hometeam", "home"])
m_away = pick_first(matchups, ["away_team", "awayteam", "away"])
if m_home is None or m_away is None:
    raise ValueError("Couldn't find home_team/away_team columns in matchups.")

m = matchups.rename(columns={m_home: "home_team", m_away: "away_team"}).copy()

strength_map = team_table.set_index("team")["strength_score"]

m["home_strength"] = m["home_team"].map(strength_map)
m["away_strength"] = m["away_team"].map(strength_map)
m["strength_diff"] = m["home_strength"] - m["away_strength"]

# Convert diff -> probability with logistic curve
# k controls spread: bigger k => more extreme probabilities
k = 1.25  # good starting value for z-score diffs
m["home_win_prob"] = 1 / (1 + np.exp(-k * m["strength_diff"]))

# Clamp to avoid silly 0/1
m["home_win_prob"] = m["home_win_prob"].clip(0.05, 0.95)

matchup_probs = m[["home_team", "away_team", "home_win_prob"]].sort_values("home_win_prob", ascending=False)

print("\nRound 1 Matchup Probabilities (Home win):")
display(matchup_probs)

# ---------- 8) Save outputs ----------
power_rankings[["rank", "team", "strength_score"]].to_csv("power_rankings_1a.csv", index=False)
matchup_probs.to_csv("round1_home_win_probs_1a.csv", index=False)

print("\nSaved:")
print(" - power_rankings_1a.csv")
print(" - round1_home_win_probs_1a.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'data/whl_2025 (1).xlsx'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')